# B2: EfficientNet-B3 Supervised Baseline

**Leaf-JEPA IRP** | Stage 3 — Baseline Establishment

## Rationale

B2 is the **parameter-efficient CNN baseline**. EfficientNet-B3 achieves comparable accuracy
to ResNet-50 with roughly half the parameters (12.2M vs 25.6M). It answers: if a small
supervised CNN already performs well, is the SSL + PEFT overhead justified?

This is the anchor for **RQ2** (parameter efficiency) and **RQ5** (PEFT vs supervised CNNs).

**Important**: Uses identical preprocessing, transforms, splits, and evaluation protocol as B1.
The ONLY difference is the model architecture.

## Iniitialization

In [1]:
# Setup and hyperparameter documentation
import sys, json, time

import numpy as np
import torch
import torch.nn as nn
from torch.amp import GradScaler
from torch.utils.data import DataLoader
import timm
import sys, warnings

from pathlib import Path


PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from stage3_baseline_establishment.config_stage3 import *
from stage3_baseline_establishment.baseline_utils import (
    seed_everything, PlantVillageDataset, load_split,
    train_one_epoch, evaluate, EarlyStopping, save_results,
    plot_confusion_matrix, label_efficiency_sweep
)

from stage2_dataset_preparation.outputs.augmentation.transforms import (
    get_pretrain_transform, get_eval_transform, get_finetune_transform
)

verify_config()
seed_everything(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

hyperparams = {
    "baseline": "B2",
    "model": "EfficientNet-B3 (timm, ImageNet pretrained)",
    "total_params": "12.2M",
    "optimiser": "AdamW",
    "backbone_lr": CNN_LR,
    "head_lr": CNN_HEAD_LR,
    "weight_decay": CNN_WEIGHT_DECAY,
    "batch_size": CNN_BATCH_SIZE,
    "max_epochs": CNN_EPOCHS,
    "patience": CNN_PATIENCE,
    "amp": True,
    "image_size": IMAGE_CROP,
    "note": "Using 224x224 input (not EfficientNet native 300) for fair ViT comparison",
}
BASELINES_DIR.mkdir(parents=True, exist_ok=True)
save_results(hyperparams, BASELINES_DIR / "B2_hyperparams.json")
print("done setup")

  ✅  All config checks passed.
Device: cuda
  Saved: /workspace/leaf-jepa/stage3_baseline_establishment/outputs/baselines/B2_hyperparams.json
done setup


## Full Training

In [2]:
# Full training
import wandb

# ── Load Stage 2 splits ──
train_paths, train_labels, class_names = load_split(SPLITS_DIR / "train_split.json")
val_paths, val_labels, _ = load_split(SPLITS_DIR / "val_split.json")
test_paths, test_labels, _ = load_split(SPLITS_DIR / "test_split.json")

train_tf = get_pretrain_transform()
eval_tf  = get_eval_transform()

train_ds = PlantVillageDataset(train_paths, train_labels, transform=train_tf)
val_ds   = PlantVillageDataset(val_paths, val_labels, transform=eval_tf)
test_ds  = PlantVillageDataset(test_paths, test_labels, transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=CNN_BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=CNN_BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=CNN_BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

def build_efficientnet():
    model = timm.create_model("efficientnet_b3", pretrained=True, num_classes=NUM_CLASSES)
    # timm creates the classifier automatically; get param groups
    classifier_params = list(model.classifier.parameters())
    classifier_ids = {id(p) for p in classifier_params}
    backbone_params = [p for p in model.parameters() if id(p) not in classifier_ids]

    param_groups = [
        {"params": backbone_params, "lr": CNN_LR},
        {"params": classifier_params, "lr": CNN_HEAD_LR},
    ]
    return model, param_groups

# ── Train on 100% labels ──
seed_everything(RANDOM_SEED)
model, param_groups = build_efficientnet()
model = model.to(device)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"EfficientNet-B3 trainable params: {n_trainable:,}")

criterion = nn.CrossEntropyLoss()
optimiser = torch.optim.AdamW(param_groups, weight_decay=CNN_WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=CNN_EPOCHS)
scaler = GradScaler(enabled=torch.cuda.is_available())
es = EarlyStopping(patience=CNN_PATIENCE)

if WANDB_ENTITY:
    os.environ["WANDB__SERVICE_WAIT"] = "10"
    os.environ["WANDB_DISABLED"] = "true"
    try:
        wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                   name="B2-EfficientNet-full", config=hyperparams)
    except:
        print("WandB init failed — training without logging")
        WANDB_ENTITY = False


print("\nTraining B2 EfficientNet-B3 (100% labels)...")
t0 = time.time()
for epoch in range(CNN_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, criterion,
                                 optimiser, scaler, device)
    val_result = evaluate(model, val_loader, device)
    scheduler.step()

    if WANDB_ENTITY:
        wandb.log({"train_loss": train_loss,
                    "val_macro_f1": val_result["macro_f1"],
                    "val_accuracy": val_result["accuracy"], "epoch": epoch})

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d} | loss={train_loss:.4f} | "
              f"val_F1={val_result['macro_f1']:.4f}")

    es.step(val_result["macro_f1"], model)
    if es.stopped:
        print(f"  Early stop at epoch {epoch+1}")
        break

elapsed = time.time() - t0
es.load_best(model)

test_result = evaluate(model, test_loader, device)
print(f"\n  B2 Test macro-F1:  {test_result['macro_f1']:.4f}")
print(f"  B2 Test accuracy:  {test_result['accuracy']:.4f}")

if WANDB_ENTITY:
    wandb.log({"test_macro_f1": test_result["macro_f1"],
               "test_accuracy": test_result["accuracy"]})
    wandb.finish()

# Save aggregate results
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
b2_aggregate = {
    "baseline": "B2", "model": "EfficientNet-B3",
    "macro_f1": test_result["macro_f1"],
    "accuracy": test_result["accuracy"],
    "per_class_f1": test_result["per_class_f1"],
    "training_time_s": elapsed,
    "best_val_f1": float(es.best_score),
    "trainable_params": n_trainable,
}
save_results(b2_aggregate, BASELINES_DIR / "B2_aggregate.json")
plot_confusion_matrix(
    np.array(test_result["confusion_matrix"]),
    class_names, FIGURES_DIR / f"B2_confusion_matrix_seed{RANDOM_SEED}.png",
    title=f"B2 EfficientNet-B3 | F1={test_result['macro_f1']:.3f}"
)

del model, optimiser, scheduler, scaler
torch.cuda.empty_cache()

EfficientNet-B3 trainable params: 10,754,638


/tmp/ipykernel_15307/1332688525.py:46: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=torch.cuda.is_available())
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: muh-haleef02 (muh-haleef02-inform) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Training B2 EfficientNet-B3 (100% labels)...


  Epoch   1 | loss=0.6260 | val_F1=0.8053


  Epoch   5 | loss=0.0664 | val_F1=0.9742


...

  Epoch  35 | loss=0.0104 | val_F1=0.9934


  Early stop at epoch 37



  B2 Test macro-F1:  0.9923
  B2 Test accuracy:  0.9951


epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
test_accuracy,▁
test_macro_f1,▁
train_loss,█▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_accuracy,▁▄▆▆▇▇▇▇▇████▇██▇████████████████████
val_macro_f1,▁▄▆▆▇▇▇▇▇████▇▇█▇████████████████████
epoch,36
test_accuracy,0.99509
test_macro_f1,0.99234
train_loss,0.00819
val_accuracy,0.99497


  Saved: /workspace/leaf-jepa/stage3_baseline_establishment/outputs/baselines/B2_aggregate.json
  Saved: /workspace/leaf-jepa/stage3_baseline_establishment/outputs/figures/B2_confusion_matrix_seed42.png


## Label efficiency sweep


In [ ]:
# ── Label efficiency sweep ──
print("\n" + "=" * 60)
print("  B2 LABEL EFFICIENCY SWEEP")
print("=" * 60)

# ── Load Stage 2 splits ──
train_paths, train_labels, class_names = load_split(SPLITS_DIR / "train_split.json")
val_paths, val_labels, _ = load_split(SPLITS_DIR / "val_split.json")
test_paths, test_labels, _ = load_split(SPLITS_DIR / "test_split.json")

train_tf = get_pretrain_transform()
eval_tf  = get_eval_transform()

train_ds = PlantVillageDataset(train_paths, train_labels, transform=train_tf)
val_ds   = PlantVillageDataset(val_paths, val_labels, transform=eval_tf)
test_ds  = PlantVillageDataset(test_paths, test_labels, transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=CNN_BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=CNN_BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=CNN_BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

def build_efficientnet():
    model = timm.create_model("efficientnet_b3", pretrained=True, num_classes=NUM_CLASSES)
    # timm creates the classifier automatically; get param groups
    classifier_params = list(model.classifier.parameters())
    classifier_ids = {id(p) for p in classifier_params}
    backbone_params = [p for p in model.parameters() if id(p) not in classifier_ids]

    param_groups = [
        {"params": backbone_params, "lr": CNN_LR},
        {"params": classifier_params, "lr": CNN_HEAD_LR},
    ]
    return model, param_groups
    
label_efficiency_sweep(
    model_factory=build_efficientnet,
    splits_dir=SPLITS_DIR,
    val_paths=val_paths, val_labels=val_labels,
    test_paths=test_paths, test_labels=test_labels,
    class_names=class_names,
    fractions=[f for f in LABEL_FRACTIONS if f < 1.0],
    seeds=SUBSET_SEEDS,
    norm_mean=NORM_MEAN, norm_std=NORM_STD,
    batch_size=CNN_BATCH_SIZE,
    lr=CNN_LR, head_lr=CNN_HEAD_LR,
    weight_decay=CNN_WEIGHT_DECAY,
    epochs=CNN_EPOCHS, patience=CNN_PATIENCE,
    device=device,
    baseline_id="B2",
    baselines_dir=BASELINES_DIR, figures_dir=FIGURES_DIR,
    wandb_project=WANDB_PROJECT if WANDB_ENTITY else None,
    wandb_entity=WANDB_ENTITY,
)
print("\nB2 complete.")


  B2 LABEL EFFICIENCY SWEEP

  B2-frac0.50-seed42


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


  Trainable params: 10,754,638
  Training samples: 19,006


wandb: Currently logged in as: muh-haleef02 (muh-haleef02-inform) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


  Training:   6%|█▋                            | 17/296 [00:21<03:51,  1.20it/s, loss=3.5048]

## Critical Analysis

B2 EfficientNet-B3 serves as the parameter-efficient CNN anchor:

1. **B2 vs B1** — If B2 matches B1 with half the parameters, this shows diminishing returns from larger supervised models and strengthens the case for parameter-efficient approaches.

2. **Label efficiency comparison** — B2's curve alongside B1's reveals whether architectural efficiency helps in low-label regimes.

3. **Stage 5 comparison** — PEFT methods (LoRA ~0.3M, VPT ~0.05M) must be compared against B2's 12.2M to justify the SSL overhead.